# Observer.ai: LoRA fine-tuning Kev-0.8B on labeled agent traces

Fine-tunes the released **Kev-0.8B** checkpoint on your own labeled traces, then checks whether it actually beats the zero-shot model as a sampler.

**Flow**
1. Load labeled traces and build trace slices (the `state`)
2. Convert to Kev's training JSONL and split train / held-out
3. Zero-shot baseline: score held-out traces with released Kev-0.8B
4. LoRA fine-tune with `kev.train --init_from` (adapter + pointer head, base frozen)
5. Score held-out with the fine-tuned run and compare: AUROC and normalized recall at 5%
6. Fit a temperature per question type and save a model manifest for the Go engine


In [27]:
import hashlib, json, math, random, subprocess, time
from pathlib import Path

import numpy as np
import requests
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score

# ---- config ----
TRACES_PATH  = Path("data/eval/labeled_traces.jsonl")   # your labeled traces (schema below)
WORK_DIR     = Path("runs/observer-kev-0.8b-ft").resolve()
BASE_MODEL   = "Qwen/Qwen3.5-0.8B-Base"
INIT_FROM    = "jaredpalmer/kev-0.8b"
DEVICE       = "mps"                                # Apple Silicon; confirm with kev.train --help
SAMPLE_RATE  = 0.05                                 # the 5% budget we evaluate at
HELDOUT_FRAC = 0.2
MAX_TOKENS   = 8192
SEED         = 7

KEV_PATH     = Path("/Users/samuelabolo/kode/hobby/kev")

ZERO_SHOT_URL  = "http://127.0.0.1:8009/v1/systemone"   # released Kev-0.8B
FINETUNED_URL  = "http://127.0.0.1:8010/v1/systemone"   # your fine-tuned run

random.seed(SEED)
np.random.seed(SEED)
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load labeled traces

Expected schema, one JSON object per line (produced by your ReAct agent + gold-answer labeller):

```json
{
  "trace_id": "7f3a91c2e8",
  "question": "Which river flows through the capital of France?",
  "tool_calls": [{"name": "search", "args": "capital of France", "status": "ok"}],
  "evidence": ["The Eiffel Tower is in Paris, France."],
  "final_answer": "The Rhône flows through the capital, Lyon.",
  "label": "wrong"
}
```

`label` is one of `correct`, `partial`, `wrong`.

In [28]:
def load_traces(path):
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    bad = [r["trace_id"] for r in rows if r["label"] not in {"correct", "partial", "wrong"}]
    assert not bad, f"unknown labels on {len(bad)} traces, e.g. {bad[:3]}"
    return rows

traces = load_traces(TRACES_PATH)
labels = [t["label"] for t in traces]
print(f"{len(traces)} traces:", {k: labels.count(k) for k in ("correct", "partial", "wrong")})

4499 traces: {'correct': 1729, 'partial': 625, 'wrong': 2145}


## 2. Build trace slices

The `state` is the slice the Go engine will send: question, tool calls, top evidence and the final answer.

In [29]:
MAX_EVIDENCE = 3
MAX_EVIDENCE_CHARS = 1500

def build_state(t):
    tools = "\n".join(
        f"{i}. {c['name']}({c.get('args', '')!r}) -> {c.get('status', 'ok')}"
        for i, c in enumerate(t.get("tool_calls", []), 1)
    ) or "(none)"
    evidence = "\n".join(
        f"[{i}] {e[:MAX_EVIDENCE_CHARS]}" for i, e in enumerate(t.get("evidence", [])[:MAX_EVIDENCE], 1)
    ) or "(none)"
    return (
        f"USER QUESTION:\n{t['question']}\n\n"
        f"TOOL CALLS:\n{tools}\n\n"
        f"EVIDENCE:\n{evidence}\n\n"
        f"FINAL ANSWER:\n{t['final_answer']}"
    )

print(build_state(traces[0]))

USER QUESTION:
What was the estimated population of the city that Luis Villalta Aquino was from in 2012?

TOOL CALLS:
1. search({'query': 'Luis Villalta Aquino city 2012 population'}) -> ok

EVIDENCE:
[1] Luis Villalta Aquino (October 2, 1969, Lima, Peru – March 3, 2004, Pompano Beach, Florida) was a professional boxer, who was nicknamed "El Puma" during his career.
[2] San Luis Potosí, commonly called SLP or simply San Luis, is the capital and the most populous city of the Mexican state of San Luis Potosí. The city lies at an elevation of 1850 m . It has an estimated population of 735,886 in the city proper and a population of approximately 1,021,688 in its metropolitan area, formed with the neighbour city of Soledad de Graciano Sánchez and some other small townships inside the urban area, which makes the metropolitan area of Greater San Luis Potosí the eleventh largest in Mexico.
[3] Pompano Beach is a city in Broward County, Florida, United States, along the coast of the Atlantic Oc

In [30]:
# Drop slices that would not fit the validated context window
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
lengths = np.array([len(tok(build_state(t))["input_ids"]) for t in traces])
print(f"tokens per slice: median {int(np.median(lengths))}, p95 {int(np.percentile(lengths, 95))}, max {lengths.max()}")

headroom = 512  # room for the questions themselves
keep = lengths <= MAX_TOKENS - headroom
traces = [t for t, k in zip(traces, keep) if k]
print(f"kept {len(traces)} traces, dropped {int((~keep).sum())} over the limit")

tokens per slice: median 394, p95 629, max 1013
kept 4499 traces, dropped 0 over the limit


## 3. Questions and Kev training format

Labels follow Kev's format: `noul` takes `true`/`false`, `score` takes the level index from 0, `choice` takes the option name.

**Primary signal:** `answer_wrong` (noul). Partial answers count as wrong, matching the main metric.

In [31]:
QUESTIONS = {
    "answer_wrong": {
        "type": "noul",
        "instructions": "Is the final answer wrong or only partially correct for the user's question?",
    },
    "answers_question": {
        "type": "score",
        "instructions": "How completely and correctly does the final answer address the user's question?",
        "criteria": ["Wrong", "Partially correct", "Fully correct"],
    },
}
SCORE_LEVEL = {"wrong": 0, "partial": 1, "correct": 2}

def to_kev_record(t):
    q = json.loads(json.dumps(QUESTIONS))  # deep copy
    q["answer_wrong"]["label"] = t["label"] != "correct"
    q["answers_question"]["label"] = SCORE_LEVEL[t["label"]]
    return {"state": build_state(t), "questions": q}

def is_wrong(t):
    return t["label"] != "correct"

# Stratified split so held-out keeps the same base rate of wrong answers
wrong   = [t for t in traces if is_wrong(t)]
correct = [t for t in traces if not is_wrong(t)]
random.shuffle(wrong); random.shuffle(correct)
n_hw, n_hc = int(len(wrong) * HELDOUT_FRAC), int(len(correct) * HELDOUT_FRAC)
heldout = wrong[:n_hw] + correct[:n_hc]
train   = wrong[n_hw:] + correct[n_hc:]
random.shuffle(train)

def write_jsonl(path, rows):
    path.write_text("\n".join(json.dumps(to_kev_record(t)) for t in rows) + "\n")

TRAIN_JSONL, HELDOUT_JSONL = WORK_DIR / "train.jsonl", WORK_DIR / "heldout.jsonl"
write_jsonl(TRAIN_JSONL, train)
write_jsonl(HELDOUT_JSONL, heldout)

base_rate = np.mean([is_wrong(t) for t in heldout])
print(f"train {len(train)}, held-out {len(heldout)}, held-out base rate of wrong answers {base_rate:.1%}")

train 3600, held-out 899, held-out base rate of wrong answers 61.6%


## 4. Scoring helpers and sampler metrics

**Normalized recall** = (sampler − random) / (oracle − random), at a fixed sample budget.
- sampler: share of wrong answers that land in the top-k by p(wrong)
- random: expected share caught by random sampling (= the sample rate)
- oracle: best possible with gold labels = min(1, k / number of wrong answers)

In [32]:
def score_traces(url, rows, timeout=60):
    p_wrong, latencies = [], []
    for t in rows:
        body = {"model": "kev-latest", "state": build_state(t), "questions": QUESTIONS}
        t0 = time.perf_counter()
        r = requests.post(url, json=body, timeout=timeout)
        r.raise_for_status()
        latencies.append((time.perf_counter() - t0) * 1000)
        p_wrong.append(r.json()["answers"]["answer_wrong"]["noul"])
    return np.array(p_wrong), np.array(latencies)

def sampler_metrics(p_wrong, y_wrong, rate=SAMPLE_RATE):
    y = np.asarray(y_wrong, dtype=bool)
    n, n_wrong = len(y), int(y.sum())
    k = max(1, math.ceil(rate * n))
    top_k = np.argsort(-p_wrong)[:k]
    sampler = y[top_k].sum() / n_wrong
    random_r = k / n
    oracle = min(1.0, k / n_wrong)
    return {
        "auroc": roc_auc_score(y, p_wrong),
        "recall_at_rate": sampler,
        "random_recall": random_r,
        "oracle_recall": oracle,
        "normalized_recall": (sampler - random_r) / (oracle - random_r),
    }

y_heldout = np.array([is_wrong(t) for t in heldout])

In [33]:
## Establish baseline with zero-shot run
p_zero, lat_zero = score_traces(ZERO_SHOT_URL, heldout)
zero_shot = sampler_metrics(p_zero, y_heldout)
print("zero-shot:", {k: round(v, 3) for k, v in zero_shot.items()})
print(f"latency p50 {np.percentile(lat_zero, 50):.0f} ms, p95 {np.percentile(lat_zero, 95):.0f} ms")

zero-shot: {'auroc': 0.456, 'recall_at_rate': np.float64(0.043), 'random_recall': 0.05, 'oracle_recall': 0.081, 'normalized_recall': np.float64(-0.216)}
latency p50 93 ms, p95 127 ms


## 5. LoRA fine-tune from the released checkpoint

`--init_from` loads Kev's LoRA adapter and pointer head, so you continue from its decision skills instead of starting from the raw base. The Qwen base stays frozen.

Stop the step 5 server first if memory is tight. Training on an M3 Max is slower than CUDA, so start with 1 epoch.

In [40]:
RUN_DIR = KEV_PATH / "run"
train_cmd = [
    "uv", "run", "python", "-m", "kev.train",
    "--data", str(TRAIN_JSONL),
    "--base", BASE_MODEL,
    "--init_from", INIT_FROM,
    "--epochs", "1",
    "--lr", "2e-5",
    "--batch", "1",
    "--accum", "8",
    "--dtype", "fp32",
    "--device", DEVICE,
    "--out", str(RUN_DIR),
]
print(" ".join(train_cmd))

t0 = time.time()
subprocess.run(train_cmd, cwd=KEV_PATH, check=True)
print(f"training took {(time.time() - t0) / 60:.1f} min")

uv run python -m kev.train --data /Users/samuelabolo/kode/hobby/observer-ai/finetune/notebooks/runs/observer-kev-0.8b-ft/train.jsonl --base Qwen/Qwen3.5-0.8B-Base --init_from jaredpalmer/kev-0.8b --epochs 1 --lr 2e-5 --batch 1 --accum 8 --dtype fp32 --device mps --out /Users/samuelabolo/kode/hobby/kev/run


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 797.16it/s]


delta: warm start from /Users/samuelabolo/.cache/huggingface/hub/models--jaredpalmer--kev-0.8b/snapshots/bf75a6a8848ea6960ff2ed108d9ed44c2941174f: 372 lora tensors and the pointer head loaded
device=mps world=1 trainable params=11.3M
dropped 1928 of 3600 records that exceed the training context (384 state / 1024 branch / 2048 packed tokens)


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


1672 training requests (holdout=[]), questions by type {'noul': 1672, 'score': 1672}
ep0 step 10/209 loss 1.189 kl 0.000 anchor 0.000 1.655s/rec
ep0 step 20/209 loss 0.954 kl 0.000 anchor 0.000 1.603s/rec
ep0 step 30/209 loss 0.919 kl 0.000 anchor 0.000 1.586s/rec
ep0 step 40/209 loss 0.833 kl 0.000 anchor 0.000 1.589s/rec
ep0 step 50/209 loss 0.824 kl 0.000 anchor 0.000 1.603s/rec
ep0 step 60/209 loss 0.708 kl 0.000 anchor 0.000 1.596s/rec
ep0 step 70/209 loss 0.907 kl 0.000 anchor 0.000 1.600s/rec
ep0 step 80/209 loss 0.838 kl 0.000 anchor 0.000 1.593s/rec
ep0 step 90/209 loss 0.770 kl 0.000 anchor 0.000 1.598s/rec
ep0 step 100/209 loss 0.774 kl 0.000 anchor 0.000 1.591s/rec
ep0 step 110/209 loss 0.778 kl 0.000 anchor 0.000 1.589s/rec
ep0 step 120/209 loss 0.852 kl 0.000 anchor 0.000 1.587s/rec
ep0 step 130/209 loss 0.728 kl 0.000 anchor 0.000 1.580s/rec
ep0 step 140/209 loss 0.827 kl 0.000 anchor 0.000 1.583s/rec
ep0 step 150/209 loss 0.765 kl 0.000 anchor 0.000 1.579s/rec
ep0 step 

In [42]:
RUN_DIR

PosixPath('/Users/samuelabolo/kode/hobby/kev/run')

## 7. Compare as a sampler

In a terminal:

```bash
uv run --extra serve python -m kev.serve --run $RUN_DIR --port 8010
```

In [43]:
p_ft, lat_ft = score_traces(FINETUNED_URL, heldout)
finetuned = sampler_metrics(p_ft, y_heldout)

print(f"{'metric':<20}{'zero-shot':>12}{'fine-tuned':>12}")
for k in zero_shot:
    print(f"{k:<20}{zero_shot[k]:>12.3f}{finetuned[k]:>12.3f}")

metric                 zero-shot  fine-tuned
auroc                      0.456       0.714
recall_at_rate             0.043       0.078
random_recall              0.050       0.050
oracle_recall              0.081       0.081
normalized_recall         -0.216       0.884


**Reading the result**
- AUROC near 0.5 means the scorer ranks no better than chance
- If fine-tuned normalized recall isn't clearly above zero-shot on the held-out set, ship zero-shot plus calibration and skip fine-tuning
- Held-out is small, so treat differences of a few points as noise. Re-run with another `SEED` before concluding.

## 8. Temperature per question type

Ranking (AUROC, recall) doesn't change with temperature, but the probabilities do. The Go engine needs calibrated `p(wrong)` for the uncertainty bonus and alert thresholds.

Fit on the held-out set here for the MVP; with more data, fit on a separate calibration split.

In [44]:
def fit_temperature(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    logits = np.log(p / (1 - p))
    def nll(T):
        q = 1 / (1 + np.exp(-logits / T))
        q = np.clip(q, 1e-9, 1 - 1e-9)
        return -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))
    return minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x

def ece(p, y, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return sum(abs(p[idx == b].mean() - y[idx == b].mean()) * (idx == b).mean()
               for b in range(bins) if (idx == b).any())

T = fit_temperature(p_ft, y_heldout.astype(float))
logits = np.log(np.clip(p_ft, 1e-6, 1 - 1e-6) / np.clip(1 - p_ft, 1e-6, 1))
p_cal = 1 / (1 + np.exp(-logits / T))
print(f"temperature {T:.2f}; ECE before {ece(p_ft, y_heldout):.3f}, after {ece(p_cal, y_heldout):.3f}")

temperature 0.86; ECE before 0.046, after 0.046


## 9. Model manifest for the Go engine

The engine reads this to know which run it is talking to, how to calibrate, and what quality to expect.

In [45]:
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]

manifest = {
    "model": "observer-kev-0.8b-ft",
    "run_dir": str(RUN_DIR),
    "base": BASE_MODEL,
    "init_from": INIT_FROM,
    "train_data_sha": file_hash(TRAIN_JSONL),
    "questions": QUESTIONS,
    "temperatures": {"answer_wrong": round(float(T), 4)},
    "heldout": {"n": len(heldout), "base_rate": round(float(base_rate), 4)},
    "metrics": {"zero_shot": zero_shot, "finetuned": finetuned},
    "latency_ms": {"p50": float(np.percentile(lat_ft, 50)), "p95": float(np.percentile(lat_ft, 95))},
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
(WORK_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=float))
print(f"wrote {WORK_DIR / 'manifest.json'}")

wrote /Users/samuelabolo/kode/hobby/observer-ai/finetune/notebooks/runs/observer-kev-0.8b-ft/manifest.json
